In [1]:
import pandas as pd
import re
from pathlib import Path

DATA_DIR = Path("data")
INPUT_PATH = DATA_DIR / "ai_news_corpus_for_rag.csv"
OUTPUT_PATH = DATA_DIR / "ai_sentiment_rich_corpus.csv"

df = pd.read_csv(INPUT_PATH)
df.head()
print(df.shape)


(463, 7)


In [2]:
sentiment_pattern = re.compile(
    r"\b("
    r"fear|fears|fearful|"
    r"concern|concerns|worry|worries|alarming|"
    r"criticize|criticism|criticised|criticized|backlash|"
    r"controversy|controversial|"
    r"threat|threaten|threatening|"
    r"risk|risky|danger|dangerous|"
    r"panic|outrage|"
    r"catastrophic|collapse|"
    r"praise|praised|promising|hope|hopeful|breakthrough|"
    r"support|supported|oppose|opposed|resistance|"
    r"celebrate|celebrated|optimistic|"
    r"warn|warning|warned"
    r")\b",
    flags=re.IGNORECASE
)


In [3]:
df["combined_text"] = (
    df["title"].fillna("") + " " +
    df["description"].fillna("") + " " +
    df["clean_content"].fillna("")
)

keyword_mask = df["combined_text"].str.contains(sentiment_pattern, na=False)
df_kw = df[keyword_mask].copy()
print("Articles with sentiment keywords:", len(df_kw))


Articles with sentiment keywords: 364


/var/folders/xv/rj070sz972sf0vvfm1w4m5ph0000gn/T/ipykernel_83442/3519284588.py:7: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  keyword_mask = df["combined_text"].str.contains(sentiment_pattern, na=False)


In [4]:
label_mask = df["sentiment"].isin([1, 2, 4, 5])
df_lab = df[label_mask].copy()
print("Articles with strong sentiment labels:", len(df_lab))


Articles with strong sentiment labels: 453


In [6]:
df_sentiment_rich = pd.concat([df_kw, df_lab]).drop_duplicates(subset=["article_id"])
df_sentiment_rich = df_sentiment_rich.reset_index(drop=True)

print("Final sentiment-rich AI articles:", len(df_sentiment_rich))
df_sentiment_rich.head()


Final sentiment-rich AI articles: 460


,article_id,source,title,description,content,clean_content,sentiment,combined_text
0,1,Al Jazeera,What is the political agenda of artificial int...,"May 17, 2023 ... We do not believe that it cou...",Could AI single-handedly decide the course of ...,Could AI single-handedly decide the course of ...,1,What is the political agenda of artificial int...
1,2,Al Jazeera,Facebook reveals AI use to block 'terrorist co...,"Jun 16, 2017 ... Facebook says it has stepped ...",US company says technology used to block child...,US company says technology used to block child...,1,Facebook reveals AI use to block 'terrorist co...
2,3,Al Jazeera,Could artificial intelligence lead to world pe...,"May 30, 2017 ... Machines and artificial intel...",Can one man with terminal cancer complete his ...,Can one man with terminal cancer complete his ...,4,Could artificial intelligence lead to world pe...
3,4,Al Jazeera,Artificial intelligence without borders | US-M...,"Jun 9, 2023 ... The US-Mexico border is alread...",The US border-industrial complex has joined th...,The US border-industrial complex has joined th...,4,Artificial intelligence without borders | US-M...
4,6,Al Jazeera,"Paris AI summit: Why won't US, UK sign global ...","Feb 12, 2025 ... Why were the US and UK oppose...",While some countries want to ensure inclusion ...,While some countries want to ensure inclusion ...,4,"Paris AI summit: Why won't US, UK sign global ..."


In [7]:
df_sentiment_rich.to_csv(OUTPUT_PATH, index=False)
print("Saved sentiment-rich AI corpus to:", OUTPUT_PATH)


Saved sentiment-rich AI corpus to: data/ai_sentiment_rich_corpus.csv
